# Leakage through time — 2-level vs 3-level optimized iSWAP pulses

Simulates both pulses from the latest git pull in the **3-level Duffing model**
(9-dim Hilbert space) and tracks the leakage out of the computational subspace
at every fine time step (dt = 0.05 ns, NOT just at knot points).

Definition used at each time t:
$$\text{leakage}(t) = 1 - \tfrac{1}{4}\,\mathrm{Tr}\!\left[P_\text{comp}\,U(t)^\dagger P_\text{comp} U(t)\, P_\text{comp}\right]$$
i.e. the average probability that a computational input state has left the
computational subspace by time $t$, averaged over the 4 comp basis states.
Reported as percent.

Both pulses live on the same 0 → 150 ns grid with dt = 0.05 ns (3001 samples),
so we can plot them on a shared time axis.

In [ ]:
# This notebook is propagation-only — it does NOT use Piccolo, so we avoid
# activating src/modulate_iswap/Project.toml (whose Manifest is dev-pathed at
# /home/atkamen/Piccolo.jl on the SSH machine and breaks on the laptop).
# A temporary env with just CairoMakie + DelimitedFiles is enough.
import Pkg
Pkg.activate(; temp = true)
Pkg.add(["CairoMakie", "DelimitedFiles"])

using LinearAlgebra
using DelimitedFiles
using Printf
using CairoMakie

In [ ]:
# 3-level Duffing operators (matches robust_iswap_detuned_2MHz_150ns_5nsbuf_3level_rollout.jl)
const n_lvl = 3
b = zeros(ComplexF64, n_lvl, n_lvl)
for j in 1:n_lvl-1
    b[j, j+1] = sqrt(j)
end
const b_3  = copy(b);  const bd_3 = b_3'
const I3   = Matrix{ComplexF64}(I, n_lvl, n_lvl)
const I9   = Matrix{ComplexF64}(I, n_lvl^2, n_lvl^2)

const X3 = b_3 + bd_3
const Y3 = im * (bd_3 - b_3)
const n3 = bd_3 * b_3

const XI = kron(X3, I3); const YI = kron(Y3, I3)
const IX = kron(I3, X3); const IY = kron(I3, Y3)
const XX = kron(X3, X3); const YY = kron(Y3, Y3)
const nI = kron(n3, I3); const In = kron(I3, n3)

# Anharmonicity (matches 3-level run)
const η_anh = -2π * 0.170
const H_anh = (η_anh / 2) * (nI * (nI - I9) + In * (In - I9))

# Computational subspace within 9-dim basis order |i₁i₂⟩
# |00⟩→1, |01⟩→2, |10⟩→4, |11⟩→5  (rows/cols 3, 6-9 are leakage)
const subspace_indices = [1, 2, 4, 5]
const Dim = n_lvl^2

# Drive detunings (both runs use Δ_mw = 0)
const Δ_mw1 = 0.0
const Δ_mw2 = 0.0

println("Hilbert dim = $Dim;  comp subspace indices = $subspace_indices")
println("η/2π = $(round(η_anh/(2π)*1e3, digits=1)) MHz")

In [ ]:
# Load full-gate pulse CSVs (columns: time_ns, g_eff, u_X1, u_Y1, u_X2, u_Y2)
const RUN_2LVL = joinpath(@__DIR__, "robust_iswap_detuned_2MHz_130nsmw_5nsgauss_5nsbuf_rollout_1kiter_seed42")
const RUN_3LVL = joinpath(@__DIR__, "robust_iswap_detuned_2MHz_130nsmw_5nsgauss_5nsbuf_3lvl_170MHzanh_rollout_seed42_Q100")

function load_pulse(rundir)
    raw = readdlm(joinpath(rundir, "pulse_full_gate.csv"), ',', skipstart=1)
    return (
        t   = vec(raw[:, 1]),
        g   = vec(raw[:, 2]),
        uX1 = vec(raw[:, 3]),
        uY1 = vec(raw[:, 4]),
        uX2 = vec(raw[:, 5]),
        uY2 = vec(raw[:, 6]),
    )
end

P2 = load_pulse(RUN_2LVL)
P3 = load_pulse(RUN_3LVL)

@assert P2.t == P3.t  "Time grids must match for shared-axis plotting"
const ts = P2.t
const dt = ts[2] - ts[1]
println("Loaded $(length(ts)) samples per pulse;  dt = $dt ns;  span = $(ts[1]) → $(ts[end]) ns")

In [ ]:
# Time-dependent Hamiltonian, evaluated directly from CSV samples (no interpolation).
# Matches the gate-frame Hamiltonian used during optimization.
function H_at(P, k)
    t  = P.t[k]
    g  = P.g[k]
    uX1, uY1, uX2, uY2 = P.uX1[k], P.uY1[k], P.uX2[k], P.uY2[k]

    H = g * (XX + YY) + H_anh

    c1 = cos(Δ_mw1 * t); s1 = sin(Δ_mw1 * t)
    H += uX1 * (XI * c1 + YI * s1)
    H += uY1 * (YI * c1 - XI * s1)

    c2 = cos(Δ_mw2 * t); s2 = sin(Δ_mw2 * t)
    H += uX2 * (IX * c2 + IY * s2)
    H += uY2 * (IY * c2 - IX * s2)

    return H
end

# Step-by-step Magnus (midpoint H per interval) propagation, recording leakage at every step.
function propagate_with_leakage(P)
    N = length(P.t)
    U = Matrix{ComplexF64}(I, Dim, Dim)
    leak  = zeros(Float64, N)         # leakage(t_k)
    Fcomp = zeros(Float64, N)         # 1 - leakage = comp-subspace survival probability
    for k in 1:N
        U_sub = U[subspace_indices, subspace_indices]
        Fcomp[k] = real(tr(U_sub' * U_sub)) / 4
        leak[k]  = 1 - Fcomp[k]
        if k < N
            dt_k = P.t[k+1] - P.t[k]
            # midpoint H (average of endpoints) — 3rd-order accurate per step
            H_mid = 0.5 * (H_at(P, k) + H_at(P, k+1))
            U = exp(-im * dt_k * H_mid) * U
        end
    end
    return leak, Fcomp, U
end

println("Propagating 2-level pulse in 9-dim Duffing...")
@time leak2, Fcomp2, U2_final = propagate_with_leakage(P2)
println("Propagating 3-level pulse in 9-dim Duffing...")
@time leak3, Fcomp3, U3_final = propagate_with_leakage(P3)

# Sanity check: unitarity
@printf("Unitarity ‖U†U-I‖:  2-lvl = %.2e   3-lvl = %.2e\n",
    norm(U2_final' * U2_final - I), norm(U3_final' * U3_final - I))

In [ ]:
# Final-time leakage + fidelity to U_goal (the iSWAP minus edge contributions)
const U_iswap_4x4 = let
    σx = ComplexF64[0.0 1.0; 1.0 0.0]
    σy = ComplexF64[0.0 -im; im 0.0]
    exp(-im * π/4 * (kron(σx, σx) + kron(σy, σy)))
end

for (lbl, U, leak_arr) in [("2-lvl pulse", U2_final, leak2), ("3-lvl pulse", U3_final, leak3)]
    Usub = U[subspace_indices, subspace_indices]
    Fsub = abs2(tr(U_iswap_4x4' * Usub)) / 16
    @printf("%-12s  F(iSWAP, comp) = %.6f   final leak = %5.2f%%   max leak in flight = %5.2f%%\n",
        lbl, Fsub, leak_arr[end]*100, maximum(leak_arr)*100)
end

In [ ]:
# Plot: leakage % through time, with g(t) envelope shaded behind for context
fig = Figure(size = (1100, 700), fontsize = 18)

# Top: g(t) and microwave amplitudes (just to orient the reader)
ax1 = Axis(fig[1, 1], ylabel = "g(t)  [rad/ns]",
    title = "Pulse layout (both runs share this g(t) envelope)")
lines!(ax1, ts, P2.g; color = :black, linewidth = 2)
hidexdecorations!(ax1, grid = false)

# Middle: leakage % vs t — linear scale
ax2 = Axis(fig[2, 1], ylabel = "leakage [%]",
    title = "Leakage out of computational subspace, every dt = 0.05 ns")
lines!(ax2, ts, leak2 .* 100; color = :crimson, linewidth = 2, label = "2-level optimized pulse")
lines!(ax2, ts, leak3 .* 100; color = :forestgreen, linewidth = 2, label = "3-level optimized pulse")
axislegend(ax2; position = :lt)
hidexdecorations!(ax2, grid = false)

# Bottom: same data on log scale, percentage
ax3 = Axis(fig[3, 1], xlabel = "t  [ns]", ylabel = "leakage [%]   (log)",
    yscale = log10)
lines!(ax3, ts, max.(leak2 .* 100, 1e-8); color = :crimson, linewidth = 2, label = "2-level")
lines!(ax3, ts, max.(leak3 .* 100, 1e-8); color = :forestgreen, linewidth = 2, label = "3-level")
axislegend(ax3; position = :lt)

linkxaxes!(ax1, ax2, ax3)
rowsize!(fig.layout, 1, Relative(0.18))
rowsize!(fig.layout, 2, Relative(0.41))
rowsize!(fig.layout, 3, Relative(0.41))

save(joinpath(@__DIR__, "leakage_through_time.png"), fig)
println("Saved: ", joinpath(@__DIR__, "leakage_through_time.png"))
display(fig)

In [ ]:
# Optional: also break out per-comp-basis-state leakage at every time step.
# leakage_i(t) = ‖(I - P_comp) U(t) |i⟩‖² for each comp basis index i ∈ subspace_indices.
# This shows which input state is most vulnerable to |2⟩ excursions.

function propagate_with_per_state_leakage(P)
    N = length(P.t)
    U = Matrix{ComplexF64}(I, Dim, Dim)
    leak_per_state = zeros(Float64, length(subspace_indices), N)
    for k in 1:N
        for (j, ci) in enumerate(subspace_indices)
            col = U[:, ci]
            leak_per_state[j, k] = 1 - sum(abs2(col[ri]) for ri in subspace_indices)
        end
        if k < N
            dt_k = P.t[k+1] - P.t[k]
            H_mid = 0.5 * (H_at(P, k) + H_at(P, k+1))
            U = exp(-im * dt_k * H_mid) * U
        end
    end
    return leak_per_state
end

println("Per-state leakage breakdown...")
@time leak2_per = propagate_with_per_state_leakage(P2)
@time leak3_per = propagate_with_per_state_leakage(P3)

const state_labels = ["|00⟩", "|01⟩", "|10⟩", "|11⟩"]

fig2 = Figure(size = (1100, 650), fontsize = 17)
ax_a = Axis(fig2[1, 1], ylabel = "leakage [%]",
    title = "2-level optimized pulse — per-state leakage in 3-level Duffing")
ax_b = Axis(fig2[2, 1], xlabel = "t  [ns]", ylabel = "leakage [%]",
    title = "3-level optimized pulse — per-state leakage in 3-level Duffing")
colors = [:steelblue, :darkorange, :forestgreen, :purple]
for j in 1:4
    lines!(ax_a, ts, leak2_per[j, :] .* 100; color = colors[j], linewidth = 2, label = state_labels[j])
    lines!(ax_b, ts, leak3_per[j, :] .* 100; color = colors[j], linewidth = 2, label = state_labels[j])
end
axislegend(ax_a; position = :lt)
axislegend(ax_b; position = :lt)
linkxaxes!(ax_a, ax_b)

save(joinpath(@__DIR__, "leakage_through_time_per_state.png"), fig2)
println("Saved: ", joinpath(@__DIR__, "leakage_through_time_per_state.png"))
display(fig2)

# DRAG-corrected 2-level pulse — does the leading-order theory hold?

The 2-level pulse gave $F = 0.99990$ robust iSWAP under bare $\sigma_x, \sigma_y$ dynamics, but leaks ~6% when re-simulated in 3-level Duffing because the drives off-resonantly excite $|2\rangle$.

DRAG theory predicts that the corrected controls

$$u_X^{\text{new}} = u_X + \frac{\dot u_Y}{\eta}, \qquad u_Y^{\text{new}} = u_Y - \frac{\dot u_X}{\eta}$$

drive the 3-level system such that the projection onto the computational subspace matches the 2-level evolution, **to leading order in $1/|\eta|$**:

$$P_\text{comp}\, U_\text{3lvl}\, P_\text{comp} \approx U_\text{2lvl} + \mathcal{O}\!\left(\frac{\Omega^2}{\eta^2}\right)$$

with leakage scaling as $(\Omega/\eta)^4$ instead of $(\Omega/\eta)^2$.

**Predictions for this pulse** ($\Omega/|\eta| \approx 0.06$):
- F(iSWAP, comp) ≈ 0.996  (= 2-level F minus $\sim(\Omega/\eta)^2 \approx 4\times 10^{-3}$)
- Leakage ≈ 10⁻³ (DRAG residual + the |11⟩↔|02⟩/|20⟩ coupling channel, which DRAG doesn't touch)
- Robustness should carry over since the qubit-subspace dynamics are preserved

**Outcomes**:
- F ≈ 0.996 + leakage ~10⁻³ → DRAG works. Use as warm-start for 3-level optimizer.
- F drops well below 0.99 → 2-level pulse's derivatives are too aggressive for leading-order DRAG; would need second-order correction or smoother controls.
- F good but leakage still ~1% → coupling-driven channel dominates; only the 3-level optimizer (with leakage_constraint actually wired up) can fix it.

In [ ]:
# Central-difference derivative on the fine time grid (dt = 0.05 ns matches the CSV grid).
# Endpoints use one-sided differences.
function central_diff(y::AbstractVector, dt_grid::Real)
    n = length(y)
    dy = similar(y)
    @inbounds for i in 2:n-1
        dy[i] = (y[i+1] - y[i-1]) / (2 * dt_grid)
    end
    dy[1] = (y[2] - y[1]) / dt_grid
    dy[n] = (y[n] - y[n-1]) / dt_grid
    return dy
end

# η for the DRAG correction = the anharmonicity of the 3-level system we're simulating in.
# This is a property of the *future* simulation, not of the 2-level optimization.
const η_drag = η_anh   # -2π * 0.170 rad/ns

du_X1 = central_diff(P2.uX1, dt)
du_Y1 = central_diff(P2.uY1, dt)
du_X2 = central_diff(P2.uX2, dt)
du_Y2 = central_diff(P2.uY2, dt)

# Symmetric per-qubit DRAG: u_X → u_X + du_Y/η,  u_Y → u_Y - du_X/η
P_drag = (
    t   = P2.t,
    g   = P2.g,
    uX1 = P2.uX1 .+ du_Y1 ./ η_drag,
    uY1 = P2.uY1 .- du_X1 ./ η_drag,
    uX2 = P2.uX2 .+ du_Y2 ./ η_drag,
    uY2 = P2.uY2 .- du_X2 ./ η_drag,
)

# How big is the DRAG correction vs the bare drive on each channel?
println("Correction-vs-bare amplitude per channel (qubit 1 shown; qubit 2 similar):")
@printf("  ‖u_X1‖∞ = %.4f rad/ns;  ‖du_Y1/η‖∞ = %.4f rad/ns   (%.1f%% of bare)\n",
    maximum(abs, P2.uX1), maximum(abs, du_Y1 ./ η_drag),
    100 * maximum(abs, du_Y1 ./ η_drag) / max(maximum(abs, P2.uX1), 1e-12))
@printf("  ‖u_Y1‖∞ = %.4f rad/ns;  ‖du_X1/η‖∞ = %.4f rad/ns   (%.1f%% of bare)\n",
    maximum(abs, P2.uY1), maximum(abs, du_X1 ./ η_drag),
    100 * maximum(abs, du_X1 ./ η_drag) / max(maximum(abs, P2.uY1), 1e-12))

println("\nPropagating 2-level + analytic DRAG pulse in 9-dim Duffing...")
@time leak_drag, Fcomp_drag, U_drag_final = propagate_with_leakage(P_drag)

@printf("Unitarity ‖U†U − I‖ = %.2e\n", norm(U_drag_final' * U_drag_final - I))

# Summary line in same format as the existing summary cell
for (lbl, U, leak_arr) in [
        ("2-lvl",          U2_final,    leak2),
        ("3-lvl",          U3_final,    leak3),
        ("2-lvl + DRAG",   U_drag_final, leak_drag),
    ]
    Usub = U[subspace_indices, subspace_indices]
    Fsub = abs2(tr(U_iswap_4x4' * Usub)) / 16
    @printf("%-14s  F(iSWAP, comp) = %.6f   final leak = %5.2f%%   max leak in flight = %5.2f%%\n",
        lbl, Fsub, leak_arr[end]*100, maximum(leak_arr)*100)
end

In [ ]:
# Three-way comparison plot — 2-level (red), 3-level optimized (green), 2-level + DRAG (purple)
fig_drag = Figure(size = (1100, 750), fontsize = 18)

ax1 = Axis(fig_drag[1, 1], ylabel = "g(t)  [rad/ns]",
    title = "g(t) envelope")
lines!(ax1, ts, P2.g; color = :black, linewidth = 2)
hidexdecorations!(ax1, grid = false)

ax2 = Axis(fig_drag[2, 1], ylabel = "leakage [%]",
    title = "Leakage out of computational subspace (3-level Duffing simulation, dt = 0.05 ns)")
lines!(ax2, ts, leak2     .* 100; color = :crimson,     linewidth = 2, label = "2-level pulse (no DRAG)")
lines!(ax2, ts, leak3     .* 100; color = :forestgreen, linewidth = 2, label = "3-level optimized pulse")
lines!(ax2, ts, leak_drag .* 100; color = :purple,      linewidth = 2, label = "2-level + analytic DRAG")
axislegend(ax2; position = :lt)
hidexdecorations!(ax2, grid = false)

ax3 = Axis(fig_drag[3, 1], xlabel = "t  [ns]", ylabel = "leakage [%]   (log)",
    yscale = log10)
lines!(ax3, ts, max.(leak2     .* 100, 1e-8); color = :crimson,     linewidth = 2)
lines!(ax3, ts, max.(leak3     .* 100, 1e-8); color = :forestgreen, linewidth = 2)
lines!(ax3, ts, max.(leak_drag .* 100, 1e-8); color = :purple,      linewidth = 2)

linkxaxes!(ax1, ax2, ax3)
rowsize!(fig_drag.layout, 1, Relative(0.18))
rowsize!(fig_drag.layout, 2, Relative(0.41))
rowsize!(fig_drag.layout, 3, Relative(0.41))

save(joinpath(@__DIR__, "leakage_through_time_with_drag.png"), fig_drag)
println("Saved: ", joinpath(@__DIR__, "leakage_through_time_with_drag.png"))
display(fig_drag)

# ε-sweep robustness — fidelity sensitivity to n̂₁, n̂₂, n̂₁n̂₂ perturbations

Standard susceptibility analysis. Add a static perturbation `ε·E` to the Hamiltonian for each error operator `E ∈ {n̂₁, n̂₂, n̂₁n̂₂}`, sweep ε ∈ [−0.02, +0.02] rad/ns (≈ ±3.2 MHz frequency drift), propagate, and record the computational-subspace fidelity to iSWAP at each ε.

**Why n̂ instead of Z**: for a Duffing/transmon, qubit frequency drift `δω` shifts the Hamiltonian by `δω · n̂` (not `δω · Z` — Z is a 2-level approximation that ignores the energy shift of |2⟩). On the computational subspace and at zero leakage, `n̂ = (I − Z)/2`, so this reduces to standard Z-dephasing physics, but on populated |2⟩ it scales 2× larger.

**Comparison**: 2-level optimized pulse (red), 3-level optimized pulse (green), 2-level + analytic DRAG (purple). Six panels: top row is F vs ε, bottom row is 1−F on log scale. Narrower curves = more robust.

In [ ]:
# Build the "default" Gaussian-square pulse (no microwaves) — same shape used in the
# existing per-run default_vs_robust comparisons.
#
# Layout: Gaussian rise (4σ) → buffer-flat at g_eff (5 ns) → flat at g_eff for `flat_mw_default` ns
# → buffer-flat (5 ns) → Gaussian fall (4σ). Microwaves are zero throughout.
# `flat_mw_default` is chosen so the total ∫g(t)dt = π/4 (a bare iSWAP rotation under XX+YY).
const σ_rise              = 1.25
const buffer_flat_duration = 5.0
const buffer_duration     = 4 * σ_rise   # = 5.0 ns
const g_eff               = 2π * 0.002

# Single-edge Gaussian rise area
const A_gauss = let
    ts_loc = collect(0:dt:buffer_duration)
    gs_loc = g_eff .* exp.(-(ts_loc .- buffer_duration).^2 ./ (2 * σ_rise^2))
    sum(gs_loc) * dt
end
const A_pre   = A_gauss + g_eff * buffer_flat_duration
const θ_goal_default = π/4 - 2 * A_pre              # rotation that flat-at-g_eff MW region must provide
const flat_mw_default = θ_goal_default / g_eff      # ns of flat-at-g_eff with zero microwaves
@printf("Default pulse: flat_mw_default = %.2f ns  (gate length ≈ %.2f ns)\n",
    flat_mw_default, 2 * buffer_duration + 2 * buffer_flat_duration + flat_mw_default)

# Construct g_default(t) on the shared 0-150 ns grid; zero microwaves; zero g(t) after gate ends.
function g_default_t(t)
    mw_start = buffer_duration + buffer_flat_duration
    mw_end   = mw_start + flat_mw_default
    gate_end = mw_end + buffer_flat_duration + buffer_duration
    if t < buffer_duration
        return g_eff * exp(-(t - buffer_duration)^2 / (2 * σ_rise^2))
    elseif t <= mw_end + buffer_flat_duration
        return g_eff
    elseif t <= gate_end
        t_post = t - (mw_end + buffer_flat_duration)
        return g_eff * exp(-t_post^2 / (2 * σ_rise^2))
    else
        return 0.0
    end
end

P_default = (
    t   = ts,
    g   = [g_default_t(t) for t in ts],
    uX1 = zeros(length(ts)), uY1 = zeros(length(ts)),
    uX2 = zeros(length(ts)), uY2 = zeros(length(ts)),
)

println("Propagating default (Gaussian-square, no microwaves) in 9-dim Duffing...")
@time leak_default, _, U_default_final = propagate_with_leakage(P_default)

# Verify the default gives iSWAP at ε=0 (modulo leakage in 3-level)
let
    Usub = U_default_final[subspace_indices, subspace_indices]
    Fsub = abs2(tr(U_iswap_4x4' * Usub)) / 16
    @printf("Default: F(iSWAP, comp) = %.6f   final leak = %5.2f%%   max leak = %5.2f%%\n",
        Fsub, leak_default[end]*100, maximum(leak_default)*100)
end

In [ ]:
# F(t) — comp-subspace fidelity to a SINGLE universal reference: the 2-level pulse's
# 2-level reference evolution `Us_2lvl_ref_P2`. With one shared target, all three curves
# answer the same question and are directly comparable:
#
#   "How close is this pulse's 3-level comp-subspace evolution to what the bare 2-level
#    pulse would produce in pure Pauli dynamics?"
#
# Interpretation per curve:
#   F_noDRAG(t)  : 2-lvl pulse simulated in 3-lvl. Drops as |2⟩ population builds and Stark
#                  shifts accumulate. Final value ≈ 1 − (Ω/η)² ≈ 0.94 (plus extra from
#                  the 6% leakage subtracting amplitude from the comp block).
#   F_DRAG(t)    : DRAG-corrected pulse simulated in 3-lvl. DRAG's CLAIM is that THIS curve
#                  stays near 1 — leading-order theory predicts F_DRAG(T) ≈ 1 − (Ω/η)⁴ ≈
#                  0.996. This is the actual test of DRAG.
#   F_3lvl(t)    : 3-lvl optimizer's controls simulated in 3-lvl. NOT a quality measure —
#                  the 3-lvl optimizer wasn't trying to match a 2-lvl pulse, so this curve
#                  shows how different its gate is from a bare 2-lvl pulse (large by design).
#
# IMPORTANT FIX: previously F_DRAG compared against `Us_2lvl_ref_Pdrag` (running the
# DRAG-corrected controls through 2-lvl dynamics), which is the wrong reference. The
# DRAG-corrected controls don't produce an iSWAP in bare 2-lvl dynamics — the derivative
# correction terms add unwanted rotation when there's no |2⟩ to cancel against. The
# correct DRAG target is `Us_2lvl_ref_P2` (the original 2-lvl pulse's 2-lvl evolution).

# 2-level (Pauli) operators
const σx2 = ComplexF64[0 1; 1 0]
const σy2 = ComplexF64[0 -im; im 0]
const I2  = Matrix{ComplexF64}(I, 2, 2)
const XI_2 = kron(σx2, I2); const YI_2 = kron(σy2, I2)
const IX_2 = kron(I2, σx2); const IY_2 = kron(I2, σy2)
const XX_2 = kron(σx2, σx2); const YY_2 = kron(σy2, σy2)

function H_2lvl_at(P, k)
    g = P.g[k]
    H = g * (XX_2 + YY_2)
    H += P.uX1[k] * XI_2 + P.uY1[k] * YI_2 + P.uX2[k] * IX_2 + P.uY2[k] * IY_2
    return H
end

function propagate_2lvl_with_history(P)
    N = length(P.t)
    U = Matrix{ComplexF64}(I, 4, 4)
    Us = Vector{Matrix{ComplexF64}}(undef, N)
    Us[1] = U
    for k in 1:N-1
        dt_k = P.t[k+1] - P.t[k]
        H_mid = 0.5 * (H_2lvl_at(P, k) + H_2lvl_at(P, k+1))
        U = exp(-im * dt_k * H_mid) * U
        Us[k+1] = U
    end
    return Us
end

function propagate_3lvl_with_history(P)
    N = length(P.t)
    U = Matrix{ComplexF64}(I, Dim, Dim)
    Us = Vector{Matrix{ComplexF64}}(undef, N)
    Us[1] = U
    for k in 1:N-1
        dt_k = P.t[k+1] - P.t[k]
        H_mid = 0.5 * (H_at(P, k) + H_at(P, k+1))
        U = exp(-im * dt_k * H_mid) * U
        Us[k+1] = U
    end
    return Us
end

# F(t) = |Tr(U_ref(t)† U_sub(t))|² / 16
function F_through_time(Us_3lvl::Vector{<:AbstractMatrix}, Us_2lvl_ref::Vector{<:AbstractMatrix})
    @assert length(Us_3lvl) == length(Us_2lvl_ref)
    N = length(Us_3lvl)
    F = zeros(Float64, N)
    for k in 1:N
        U_sub = Us_3lvl[k][subspace_indices, subspace_indices]
        U_ref = Us_2lvl_ref[k]
        F[k]  = abs2(tr(U_ref' * U_sub)) / 16
    end
    return F
end

println("Building 2-level reference trajectory (P2 — the universal target) ...")
@time Us_2lvl_ref_P2 = propagate_2lvl_with_history(P2)

println("Building 3-level trajectories ...")
@time Us_2lvl_in_3lvl = propagate_3lvl_with_history(P2)
@time Us_3lvl_in_3lvl = propagate_3lvl_with_history(P3)
@time Us_drag_in_3lvl = propagate_3lvl_with_history(P_drag)

# All three F(t) curves use the SAME reference: the 2-level pulse's 2-level evolution.
F_noDRAG = F_through_time(Us_2lvl_in_3lvl, Us_2lvl_ref_P2)
F_3lvl   = F_through_time(Us_3lvl_in_3lvl, Us_2lvl_ref_P2)   # not a quality measure
F_DRAG   = F_through_time(Us_drag_in_3lvl, Us_2lvl_ref_P2)   # the DRAG test

@printf("Final F(T) vs U_2lvl_ref(P2):\n")
@printf("  2-lvl pulse (no DRAG): %.4f   (expected ≈ 1 − (Ω/η)² + leakage subtraction)\n",
    F_noDRAG[end])
@printf("  3-lvl optimized:       %.4f   (NOT a quality measure — different gate by design)\n",
    F_3lvl[end])
@printf("  2-lvl + DRAG:          %.4f   (DRAG test — expected ≈ 0.996)\n", F_DRAG[end])

In [ ]:
# Focused comparison: leakage(t) and F(t) for 2-lvl (no DRAG), 3-lvl optimized, 2-lvl + DRAG,
# plus the default Gaussian-square pulse on the leakage panel for baseline reference.
#
# F(t) uses a SINGLE universal reference = the 2-lvl pulse's 2-lvl evolution. See cell 13
# header for what each curve actually measures. The DRAG curve (purple) is the meaningful
# test — predicted to stay ≈ 1 throughout if DRAG works at leading order.

fig_dragF = Figure(size = (1100, 750), fontsize = 18)

ax_leak = Axis(fig_dragF[1, 1], ylabel = "leakage [%]",
    title = "Leakage out of computational subspace")
lines!(ax_leak, ts, leak2        .* 100; color = :crimson,     linewidth = 2, label = "2-lvl pulse (no DRAG)")
lines!(ax_leak, ts, leak3        .* 100; color = :forestgreen, linewidth = 2, label = "3-lvl optimized")
lines!(ax_leak, ts, leak_drag    .* 100; color = :purple,      linewidth = 2, label = "2-lvl + analytic DRAG")
lines!(ax_leak, ts, leak_default .* 100; color = (:black, 0.4), linewidth = 1.5, linestyle = :dash, label = "default (no MW)")
axislegend(ax_leak; position = :lt)
hidexdecorations!(ax_leak, grid = false)

ax_F = Axis(fig_dragF[2, 1], xlabel = "t  [ns]",
    ylabel = "F(t)  vs 2-lvl pulse 2-lvl ref",
    title  = "Comp-subspace fidelity to the 2-level pulse's 2-level evolution (the DRAG target)")
lines!(ax_F, ts, F_noDRAG; color = :crimson,     linewidth = 2,
    label = "2-lvl pulse in 3-lvl (no DRAG)")
lines!(ax_F, ts, F_3lvl;   color = :forestgreen, linewidth = 2,
    label = "3-lvl optim (different gate — not a quality measure)")
lines!(ax_F, ts, F_DRAG;   color = :purple,      linewidth = 2,
    label = "2-lvl + DRAG (the test)")
hlines!(ax_F, [1.0]; color = :black, linestyle = :dash, linewidth = 1)
axislegend(ax_F; position = :lb, labelsize = 13)

linkxaxes!(ax_leak, ax_F)

save(joinpath(@__DIR__, "drag_vs_nodrag_leakage_and_fidelity.png"), fig_dragF)
println("Saved: ", joinpath(@__DIR__, "drag_vs_nodrag_leakage_and_fidelity.png"))
display(fig_dragF)

# Stretched-pulse test: half amplitude, double duration

Construct `P2_long` from the 2-lvl pulse by scaling time by 2× and halving every amplitude (g(t), u_X1, u_Y1, u_X2, u_Y2). Effect on the physics:

- All rotation areas ∫g·dt and ∫u·dt are preserved → same iSWAP target rotation in 2-level dynamics.
- Microwave Rabi rates drop 2× → **Ω/|η| drops from 0.06 to 0.03**.
- Predicted leakage: `(Ω/η)² ≈ 9×10⁻⁴` (vs 4×10⁻³ before) — a **~4× reduction** from the Ω/|η| scaling alone, plus an additional Gaussian-spectrum-shifted-to-DC factor that's hard to predict in closed form but typically helps further.
- `H_anh` doesn't scale (it's a property of the qubit, not the controls), so the |2⟩ state is still 170 MHz away — DRAG's small parameter genuinely shrinks.

Then propagate P2_long in the 9-dim Duffing and plot leakage(t) and F(t) up to 300 ns alongside the original 150 ns pulse for comparison.

In [ ]:
# Build P2_long — original 2-lvl pulse stretched 2× in time with all amplitudes halved.
# Time grid: 0 to 300 ns at dt = 0.05 ns (6001 samples).
# At new time t, sample the original pulse at t/2 (linear interpolation between CSV points),
# then scale all amplitudes by 0.5.

ts_long = collect(0.0:dt:300.0)

function _lookup_P2(t_orig::Float64)
    # Linear interp at original time t_orig ∈ [0, 150] ns.
    if t_orig < 0.0 || t_orig > P2.t[end]
        return (g=0.0, uX1=0.0, uY1=0.0, uX2=0.0, uY2=0.0)
    end
    k = clamp(searchsortedlast(P2.t, t_orig), 1, length(P2.t) - 1)
    α = (t_orig - P2.t[k]) / (P2.t[k+1] - P2.t[k])
    return (
        g   = (1-α)*P2.g[k]   + α*P2.g[k+1],
        uX1 = (1-α)*P2.uX1[k] + α*P2.uX1[k+1],
        uY1 = (1-α)*P2.uY1[k] + α*P2.uY1[k+1],
        uX2 = (1-α)*P2.uX2[k] + α*P2.uX2[k+1],
        uY2 = (1-α)*P2.uY2[k] + α*P2.uY2[k+1],
    )
end

_samples_long = [_lookup_P2(t/2) for t in ts_long]

P2_long = (
    t   = ts_long,
    g   = 0.5 .* [p.g   for p in _samples_long],
    uX1 = 0.5 .* [p.uX1 for p in _samples_long],
    uY1 = 0.5 .* [p.uY1 for p in _samples_long],
    uX2 = 0.5 .* [p.uX2 for p in _samples_long],
    uY2 = 0.5 .* [p.uY2 for p in _samples_long],
)

@printf("P2_long: %d samples spanning 0 → %.1f ns\n", length(ts_long), ts_long[end])
@printf("  Original P2 max |u_X1| = %.5f rad/ns;  P2_long max |u_X1| = %.5f rad/ns\n",
    maximum(abs, P2.uX1), maximum(abs, P2_long.uX1))
@printf("  Original P2 max g      = %.5f rad/ns;  P2_long max g      = %.5f rad/ns\n",
    maximum(P2.g), maximum(P2_long.g))
@printf("  Ω/|η| ratio: original = %.3f,  P2_long = %.3f\n",
    maximum(abs, P2.uX1) / abs(η_anh), maximum(abs, P2_long.uX1) / abs(η_anh))

println("\nPropagating P2_long in 9-dim Duffing (6001 steps; ~10-20s)...")
@time leak_long, _, U_long_final = propagate_with_leakage(P2_long)

println("\nBuilding 2-level reference for P2_long ...")
@time Us_2lvl_ref_long = propagate_2lvl_with_history(P2_long)

println("Building 3-level trajectory for F(t) on P2_long ...")
@time Us_long_in_3lvl = propagate_3lvl_with_history(P2_long)

F_long = F_through_time(Us_long_in_3lvl, Us_2lvl_ref_long)

let
    Usub = U_long_final[subspace_indices, subspace_indices]
    Fsub_iswap = abs2(tr(U_iswap_4x4' * Usub)) / 16
    @printf("\nP2_long results:\n")
    @printf("  F(iSWAP, comp)            = %.6f\n", Fsub_iswap)
    @printf("  Final leakage             = %5.2f%%\n", leak_long[end]*100)
    @printf("  Max in-flight leakage     = %5.2f%%\n", maximum(leak_long)*100)
    @printf("  F(T) vs 2-lvl ref         = %.6f\n", F_long[end])
    @printf("\n  Compare to original P2 in 3-lvl (no DRAG):\n")
    @printf("    Final leakage           = %5.2f%%\n", leak2[end]*100)
    @printf("    F(T) vs 2-lvl ref       = %.6f\n", F_noDRAG[end])
end

In [ ]:
# Plot leakage(t) and F(t) for P2_long (300 ns axis) with the original P2 (150 ns) overlaid
fig_long = Figure(size = (1200, 750), fontsize = 18)

ax_g = Axis(fig_long[1, 1], ylabel = "g(t)  [rad/ns]",
    title = "Stretched 2× / halved-amplitude pulse (P2_long) — 300 ns axis")
lines!(ax_g, P2.t,    P2.g;        color = :crimson,  linewidth = 1.5, label = "P2 (150 ns)")
lines!(ax_g, ts_long, P2_long.g;   color = :darkblue, linewidth = 2,   label = "P2_long (300 ns)")
axislegend(ax_g; position = :rt, labelsize = 13)
hidexdecorations!(ax_g, grid = false)

ax_leak = Axis(fig_long[2, 1], ylabel = "leakage [%]")
lines!(ax_leak, P2.t,    leak2     .* 100; color = :crimson,  linewidth = 2, label = "P2 (no DRAG, 150 ns)")
lines!(ax_leak, ts_long, leak_long .* 100; color = :darkblue, linewidth = 2, label = "P2_long (halved-amp, 300 ns)")
# Reference: DRAG-corrected original for context
lines!(ax_leak, P2.t,    leak_drag .* 100; color = (:purple, 0.5), linewidth = 1.5, linestyle = :dash,
    label = "P2 + DRAG (150 ns)")
axislegend(ax_leak; position = :lt, labelsize = 13)
hidexdecorations!(ax_leak, grid = false)

ax_F = Axis(fig_long[3, 1], xlabel = "t  [ns]", ylabel = "F(t)  vs 2-lvl ref")
lines!(ax_F, P2.t,    F_noDRAG; color = :crimson,  linewidth = 2, label = "P2 (no DRAG, 150 ns)")
lines!(ax_F, ts_long, F_long;   color = :darkblue, linewidth = 2, label = "P2_long (300 ns)")
lines!(ax_F, P2.t,    F_DRAG;   color = (:purple, 0.5), linewidth = 1.5, linestyle = :dash,
    label = "P2 + DRAG (150 ns)")
hlines!(ax_F, [1.0]; color = :black, linestyle = :dash, linewidth = 1)
axislegend(ax_F; position = :lb, labelsize = 13)

linkxaxes!(ax_g, ax_leak, ax_F)
rowsize!(fig_long.layout, 1, Relative(0.20))
rowsize!(fig_long.layout, 2, Relative(0.40))
rowsize!(fig_long.layout, 3, Relative(0.40))

save(joinpath(@__DIR__, "P2_long_leakage_and_F.png"), fig_long)
println("Saved: ", joinpath(@__DIR__, "P2_long_leakage_and_F.png"))
display(fig_long)

In [ ]:
# Dedicated F(t) plot for the doubled-time pulse — two stacked panels:
#   Top    : F(t) vs each pulse's 2-level reference (the DRAG-quality metric)
#   Bottom : F(t) vs the iSWAP target itself (shows gate-progress toward the target)
# Both panels span 0 → 300 ns to fit the P2_long curve.

# F(t) vs iSWAP target: |Tr(U_iswap† U_sub(t))|² / 16 at every saved knot
function F_iswap_through_time(Us_3lvl::Vector{<:AbstractMatrix})
    N = length(Us_3lvl)
    F = zeros(Float64, N)
    for k in 1:N
        U_sub = Us_3lvl[k][subspace_indices, subspace_indices]
        F[k]  = abs2(tr(U_iswap_4x4' * U_sub)) / 16
    end
    return F
end

F_iswap_long    = F_iswap_through_time(Us_long_in_3lvl)
F_iswap_P2      = F_iswap_through_time(Us_2lvl_in_3lvl)
F_iswap_DRAG    = F_iswap_through_time(Us_drag_in_3lvl)

fig_Flong = Figure(size = (1200, 700), fontsize = 18)

ax_Fref = Axis(fig_Flong[1, 1],
    ylabel = "F(t) vs 2-lvl ref",
    title  = "Doubled-time / halved-amplitude pulse: comp-subspace fidelity through time")
lines!(ax_Fref, P2.t,    F_noDRAG; color = :crimson,  linewidth = 2, label = "P2 (no DRAG, 150 ns)")
lines!(ax_Fref, P2.t,    F_DRAG;   color = :purple,   linewidth = 2, linestyle = :dash,
    label = "P2 + DRAG (150 ns)")
lines!(ax_Fref, ts_long, F_long;   color = :darkblue, linewidth = 2.5, label = "P2_long (300 ns)")
hlines!(ax_Fref, [1.0]; color = :black, linestyle = :dot, linewidth = 1)
axislegend(ax_Fref; position = :lb, labelsize = 13)
hidexdecorations!(ax_Fref, grid = false)

ax_Fiswap = Axis(fig_Flong[2, 1],
    xlabel = "t  [ns]",
    ylabel = "F(t) vs iSWAP target",
    title  = "Gate progress toward iSWAP")
lines!(ax_Fiswap, P2.t,    F_iswap_P2;   color = :crimson,  linewidth = 2, label = "P2 (no DRAG, 150 ns)")
lines!(ax_Fiswap, P2.t,    F_iswap_DRAG; color = :purple,   linewidth = 2, linestyle = :dash,
    label = "P2 + DRAG (150 ns)")
lines!(ax_Fiswap, ts_long, F_iswap_long; color = :darkblue, linewidth = 2.5, label = "P2_long (300 ns)")
hlines!(ax_Fiswap, [1.0]; color = :black, linestyle = :dot, linewidth = 1)
hlines!(ax_Fiswap, [0.5]; color = :gray,  linestyle = :dot, linewidth = 1)
text!(ax_Fiswap, 5, 0.51; text = "F(I, iSWAP) = 1/2", fontsize = 12, color = :gray)
axislegend(ax_Fiswap; position = :rc, labelsize = 13)

linkxaxes!(ax_Fref, ax_Fiswap)

# Stretch the x-axis to span the full 0 → 300 ns
xlims!(ax_Fref,   0.0, 300.0)
xlims!(ax_Fiswap, 0.0, 300.0)

@printf("Final F values:\n")
@printf("  P2      vs 2-lvl ref: %.4f,   vs iSWAP: %.4f\n", F_noDRAG[end], F_iswap_P2[end])
@printf("  P2+DRAG vs 2-lvl ref: %.4f,   vs iSWAP: %.4f\n", F_DRAG[end],   F_iswap_DRAG[end])
@printf("  P2_long vs 2-lvl ref: %.4f,   vs iSWAP: %.4f\n", F_long[end],   F_iswap_long[end])

save(joinpath(@__DIR__, "P2_long_fidelity_only.png"), fig_Flong)
println("Saved: ", joinpath(@__DIR__, "P2_long_fidelity_only.png"))
display(fig_Flong)

In [ ]:
# ε-sweep: for each pulse and each error operator, propagate with H + ε·E during the
# pulse's ACTUAL gate duration only (NOT through the post-gate padding in
# pulse_full_gate.csv). plot_results.jl propagates V_rise + U_flat + V_fall = 144.58 ns;
# the CSV runs to 150 ns, leaving ~5.4 ns of post-gate dead-time. If ε·E is applied during
# that dead time, it acts as an extra static rotation (since H_anh is zero on the comp
# subspace), shifting the susceptibility curves at ε ≠ 0.
#
# Gate-end detection is inlined into propagate_perturbed_final to avoid kernel-level
# function-name caching issues on cell re-runs.

function propagate_perturbed_final(P, ε::Real, E_op::AbstractMatrix)
    # Detect gate end: last index where g(t) is non-trivial OR any microwave channel
    # is nonzero. Don't propagate past this (post-gate padding only has H_anh which is
    # zero on the comp subspace; applying ε·E there would be an unintended static rotation).
    gmax = maximum(abs, P.g)
    g_rel_tol = 1e-6
    u_abs_tol = 1e-12
    last_g = findlast(>(g_rel_tol * gmax), P.g)
    last_u = findlast(k -> abs(P.uX1[k]) > u_abs_tol || abs(P.uY1[k]) > u_abs_tol ||
                          abs(P.uX2[k]) > u_abs_tol || abs(P.uY2[k]) > u_abs_tol,
                      eachindex(P.t))
    K = last_u === nothing ? last_g : max(last_g, last_u)

    U = Matrix{ComplexF64}(I, Dim, Dim)
    for k in 1:K-1
        dt_k = P.t[k+1] - P.t[k]
        H_mid = 0.5 * (H_at(P, k) + H_at(P, k+1)) + ε * E_op
        U = exp(-im * dt_k * H_mid) * U
    end
    return U
end

# Use plain Julia bindings (no `const`) so cell re-runs don't trip the const-redefinition
# check when the user iterates on this analysis.
nn_op = nI * In   # n̂₁ · n̂₂

error_ops = [
    ("n_1",     nI),
    ("n_2",     In),
    ("n_1·n_2", nn_op),
]

# P2_long is included so the susceptibility panels show how the doubled-time pulse compares
# to the original P2, the 3-lvl optimizer's output, DRAG, and the no-MW default.
pulses_for_sweep = [
    ("2-lvl",         P2,        :crimson),
    ("3-lvl",         P3,        :forestgreen),
    ("2-lvl + DRAG",  P_drag,    :purple),
    ("default",       P_default, :gray),
    ("P2_long",       P2_long,   :darkblue),
]

# Diagnostic: print each pulse's gate-end time (re-runs propagate_perturbed_final logic
# inline since the helper isn't a named function any more).
println("Detected gate-end times (should ≈ T_total_gate_actual per parameters.txt):")
for (p_name, P, _) in pulses_for_sweep
    gmax = maximum(abs, P.g)
    last_g = findlast(>(1e-6 * gmax), P.g)
    last_u = findlast(k -> abs(P.uX1[k]) > 1e-12 || abs(P.uY1[k]) > 1e-12 ||
                          abs(P.uX2[k]) > 1e-12 || abs(P.uY2[k]) > 1e-12,
                      eachindex(P.t))
    K = last_u === nothing ? last_g : max(last_g, last_u)
    @printf("  %-14s gate_end = %.2f ns  (sample %d/%d)\n",
        p_name, P.t[K], K, length(P.t))
end

εs = collect(range(-0.02, 0.02, length = 21))   # ±0.02 rad/ns ≈ ±3.2 MHz

# Storage: F_sweep[(pulse_name, err_name)] = vector of F vs ε
F_sweep    = Dict{Tuple{String,String}, Vector{Float64}}()
leak_sweep = Dict{Tuple{String,String}, Vector{Float64}}()

for (p_name, P, _) in pulses_for_sweep
    for (e_name, E_op) in error_ops
        Fs = zeros(length(εs))
        Ls = zeros(length(εs))
        print("Sweeping $p_name / $e_name ... ")
        t0 = time()
        for (i, ε) in enumerate(εs)
            U  = propagate_perturbed_final(P, ε, E_op)
            Us = U[subspace_indices, subspace_indices]
            Fs[i] = abs2(tr(U_iswap_4x4' * Us)) / 16
            Ls[i] = 1 - real(tr(Us' * Us)) / 4
        end
        F_sweep[(p_name, e_name)]    = Fs
        leak_sweep[(p_name, e_name)] = Ls
        @printf("done (%5.1f s,  F(ε=0)=%.4f,  max leak %.2f%%)\n",
            time() - t0, Fs[(length(εs)+1)÷2], maximum(Ls)*100)
    end
end

println("\nε-sweep complete.")

In [ ]:
# 6-panel ε-sweep plot — F (top row) and 1-F (bottom row, log) for each error operator
fig_eps = Figure(size = (1500, 800), fontsize = 17)

# Convert ε from rad/ns to MHz for the x-axis (factor 1000/(2π))
εs_MHz = εs .* 1000 ./ (2π)

err_titles = ["n̂₁ error", "n̂₂ error", "n̂₁·n̂₂ error"]

# Track axes explicitly so linkxaxes! gets only Axis blocks (not Legend/etc).
eps_axes_top = Axis[]
eps_axes_bot = Axis[]

# Top row: linear F vs ε
for (col, (e_name, _)) in enumerate(error_ops)
    ax = Axis(fig_eps[1, col], ylabel = (col == 1 ? "Fidelity" : ""),
        title = err_titles[col])
    for (p_name, _, color) in pulses_for_sweep
        lines!(ax, εs_MHz, F_sweep[(p_name, e_name)];
            color = color, linewidth = 2, label = p_name)
    end
    if col == 1
        axislegend(ax; position = :lb)
    end
    hidexdecorations!(ax, grid = false)
    push!(eps_axes_top, ax)
end

# Bottom row: log infidelity vs ε
for (col, (e_name, _)) in enumerate(error_ops)
    ax = Axis(fig_eps[2, col], xlabel = "ε  [MHz]", ylabel = (col == 1 ? "1 − Fidelity" : ""),
        yscale = log10)
    for (p_name, _, color) in pulses_for_sweep
        infid = 1 .- F_sweep[(p_name, e_name)]
        lines!(ax, εs_MHz, max.(infid, 1e-8); color = color, linewidth = 2)
    end
    push!(eps_axes_bot, ax)
end

linkxaxes!(eps_axes_top..., eps_axes_bot...)

save(joinpath(@__DIR__, "epsilon_sweep_default_vs_robust_drag.png"), fig_eps)
println("Saved: ", joinpath(@__DIR__, "epsilon_sweep_default_vs_robust_drag.png"))
display(fig_eps)

In [ ]:
# Combined-style plot — mirrors the per-run combined.png layout:
#   Top row    : 1-F vs ε (log) for n̂₁, n̂₂, n̂₁·n̂₂
#   Middle     : g(t) envelope of the focal pulse
#   Bottom     : u_X1, u_Y1, u_X2, u_Y2 of the focal pulse
#
# Focal pulse here = the 2-lvl + analytic DRAG pulse.
# Susceptibility panels overlay all four pulses so you can read off where DRAG sits
# relative to the 2-lvl, 3-lvl, and default curves.

fig_comb = Figure(size = (1300, 950), fontsize = 17)

# ---- Top row: susceptibility (1-F log) ----
top_axes = []
for (col, (e_name, _)) in enumerate(error_ops)
    ax = Axis(fig_comb[1, col],
        xlabel = "ε  [MHz]",
        ylabel = (col == 1 ? "1 − Fidelity" : ""),
        title  = "$(e_name) error",
        yscale = log10)
    for (p_name, _, color) in pulses_for_sweep
        infid = 1 .- F_sweep[(p_name, e_name)]
        ls = (p_name == "default") ? :dash : :solid
        lines!(ax, εs_MHz, max.(infid, 1e-8);
            color = color, linewidth = 2, linestyle = ls,
            label = p_name)
    end
    if col == 1
        axislegend(ax; position = :lb, labelsize = 13)
    end
    push!(top_axes, ax)
end

# Span the next two rows across all 3 columns
ax_g  = Axis(fig_comb[2, 1:3], ylabel = "g(t)  [rad/ns]",
    title = "Focal pulse: 2-lvl + analytic DRAG")
ax_u  = Axis(fig_comb[3, 1:3], xlabel = "t  [ns]", ylabel = "u  [rad/ns]")

# Detect the focal pulse's actual gate-end times from g(t) — it's been clean enough that we
# can find the falling Gaussian by looking at where g(t) drops back below 99% of g_eff for
# the FIRST time after staying high (this is the start of the falling buffer/Gaussian region).
# This works for the DRAG pulse since g(t) = P2.g exactly. For the default focal would
# need a different calculation — we're not using `flat_mw_default` here.
focal_g = P_drag.g
threshold = 0.99 * maximum(focal_g)
# Index where g first exceeds threshold (end of rise) and where it last exceeds threshold (start of fall)
rise_done_idx = findfirst(>=(threshold), focal_g)
fall_start_idx = findlast(>=(threshold), focal_g)
gate_end_idx  = findlast(>(1e-6 * maximum(focal_g)), focal_g)
mw_start_focal = ts[rise_done_idx] - buffer_flat_duration   # subtract the post-rise buffer
mw_end_focal   = ts[fall_start_idx] + buffer_flat_duration  # add the pre-fall buffer
gate_end_focal = ts[gate_end_idx]

# Shade Gaussian-edge (gray) and buffer-flat (gold) regions on both ends of the focal gate.
for ax in (ax_g, ax_u)
    # Beginning of gate
    vspan!(ax, [0.0],                  [ts[rise_done_idx] - buffer_flat_duration]; color = (:gray, 0.10))
    vspan!(ax, [mw_start_focal],       [mw_start_focal + buffer_flat_duration];    color = (:goldenrod, 0.15))
    # End of gate
    vspan!(ax, [mw_end_focal - buffer_flat_duration], [mw_end_focal];              color = (:goldenrod, 0.15))
    vspan!(ax, [mw_end_focal],         [gate_end_focal];                            color = (:gray, 0.10))
end

# Middle: g(t) envelope of the focal (DRAG) pulse
lines!(ax_g, ts, P_drag.g; color = :black, linewidth = 2)
hidexdecorations!(ax_g, grid = false)

# Bottom: DRAG-corrected microwave envelopes
lines!(ax_u, ts, P_drag.uX1; color = :crimson,     linewidth = 1.8, label = "u_X1")
lines!(ax_u, ts, P_drag.uY1; color = :orange,      linewidth = 1.8, label = "u_Y1")
lines!(ax_u, ts, P_drag.uX2; color = :forestgreen, linewidth = 1.8, label = "u_X2")
lines!(ax_u, ts, P_drag.uY2; color = :purple,      linewidth = 1.8, label = "u_Y2")
hlines!(ax_u, [-2π*0.01, 2π*0.01]; color = :gray, linestyle = :dot, linewidth = 1)
axislegend(ax_u; position = :rt, orientation = :horizontal, labelsize = 13)

linkxaxes!(ax_g, ax_u)
rowsize!(fig_comb.layout, 1, Relative(0.35))
rowsize!(fig_comb.layout, 2, Relative(0.22))
rowsize!(fig_comb.layout, 3, Relative(0.43))

save(joinpath(@__DIR__, "combined_drag.png"), fig_comb)
println("Saved: ", joinpath(@__DIR__, "combined_drag.png"))
display(fig_comb)

# Independent verification with QuantumToolbox `sesolve`

Cross-check the midpoint-Magnus values against QuantumToolbox's adaptive ODE integrator (Tsit5 with tolerance 1e-10). Same physics, different integrator — if the final F and leakage agree, the Magnus propagation is validated.

Strategy:
- Build the 3-level Duffing operators as QT `Qobj`s
- Construct a `QobjEvo` time-dependent Hamiltonian using cubic interpolation between the CSV samples for g(t) and the four microwave channels
- For each computational basis state, propagate from t=0 to t=gate_end with `sesolve` at abstol=reltol=1e-10
- Assemble the 4×4 comp-subspace block of the propagator, compute F(iSWAP) and leakage
- Compare to my Magnus values

For each pulse this is 4 sesolve calls × ~5 s/call ≈ 20 s. We'll do P3 (3-lvl), P_drag, P2.

# Ipopt iteration history

Parses the per-iter table from Ipopt's `output_file` log and plots:

- **Objective** vs iteration (log y)
- **`inf_pr`** (primal infeasibility = max constraint violation) vs iteration (log y)
- **`inf_du`** (dual infeasibility) vs iteration (log y)

The iter table has the format `iter objective inf_pr inf_du lg(mu) ||d|| lg(rg) alpha_du alpha_pr ls`; we parse columns 1–4. Lines that begin with the literal text `iter ` (the periodic re-header that Ipopt prints every 10 iters) are skipped.

**Data notes:**
- 2-level rollout-1kiter log was generated on the SSH machine but not pushed; closest local proxy is `ipopt_robust_iswap_detuned_2MHz_130nsmw_5nsgauss_5nsbuf.log` (1000-iter older direct-template run with identical g_eff, N_knots, Q_r, T_mw).
- 3-level log on disk is truncated to ~60 iters (`ipopt_2MHz_130nsmw_5nsgauss_5nsbuf_3lvl_170MHzanh_rollout_seed42.log`). The chat output reported 1000 iters / 35.8 s per iter, so the on-disk file is partial — the curve will end abruptly.
- If you `scp` either full log down later, just edit the paths in the next cell.

In [ ]:
# Parser for the Ipopt per-iter table.
# Each iter line starts with whitespace + an integer, then space-separated floats:
#   iter  objective  inf_pr  inf_du  lg(mu)  ||d||  lg(rg)  alpha_du  alpha_pr  ls
# We grab iter, objective, inf_pr, inf_du.

function parse_ipopt_iters(path::AbstractString)
    iters    = Int[]
    objs     = Float64[]
    inf_prs  = Float64[]
    inf_dus  = Float64[]
    open(path, "r") do io
        for line in eachline(io)
            stripped = lstrip(line)
            startswith(stripped, "iter ") && continue          # re-header
            isempty(stripped) && continue
            # First token must be an integer (iter index)
            toks = split(stripped)
            length(toks) < 4 && continue
            i = tryparse(Int, toks[1])
            i === nothing && continue
            obj    = tryparse(Float64, toks[2])
            inf_pr = tryparse(Float64, toks[3])
            inf_du = tryparse(Float64, toks[4])
            (obj === nothing || inf_pr === nothing || inf_du === nothing) && continue
            push!(iters, i); push!(objs, obj)
            push!(inf_prs, inf_pr); push!(inf_dus, inf_du)
        end
    end
    return (iter = iters, obj = objs, inf_pr = inf_prs, inf_du = inf_dus)
end

# Default paths — edit if you have fresher logs scp'd down.
const LOG_2LVL = joinpath(@__DIR__, "ipopt_robust_iswap_detuned_2MHz_130nsmw_5nsgauss_5nsbuf.log")
const LOG_3LVL = joinpath(@__DIR__, "ipopt_2MHz_130nsmw_5nsgauss_5nsbuf_3lvl_170MHzanh_rollout_seed42.log")

for p in (LOG_2LVL, LOG_3LVL)
    @assert isfile(p) "Log not found: $p"
end

it2 = parse_ipopt_iters(LOG_2LVL)
it3 = parse_ipopt_iters(LOG_3LVL)

@printf("2-level log: %4d iter rows  (last iter = %d, obj = %.4e, inf_pr = %.2e)\n",
    length(it2.iter), it2.iter[end], it2.obj[end], it2.inf_pr[end])
@printf("3-level log: %4d iter rows  (last iter = %d, obj = %.4e, inf_pr = %.2e)\n",
    length(it3.iter), it3.iter[end], it3.obj[end], it3.inf_pr[end])

In [ ]:
# Plot iteration history: objective, inf_pr, inf_du — log y, shared x

fig3 = Figure(size = (1100, 800), fontsize = 18)

ax_obj = Axis(fig3[1, 1], ylabel = "objective", yscale = log10,
    title = "Ipopt iteration history — 2-level vs 3-level")
lines!(ax_obj, it2.iter, max.(it2.obj, 1e-12); color = :crimson,    linewidth = 2, label = "2-level (proxy: direct, 1000 iter)")
lines!(ax_obj, it3.iter, max.(it3.obj, 1e-12); color = :forestgreen, linewidth = 2, label = "3-level rollout (truncated)")
axislegend(ax_obj; position = :rt)
hidexdecorations!(ax_obj, grid = false)

ax_pr = Axis(fig3[2, 1], ylabel = "inf_pr  (constraint violation)", yscale = log10)
lines!(ax_pr, it2.iter, max.(it2.inf_pr, 1e-12); color = :crimson,    linewidth = 2)
lines!(ax_pr, it3.iter, max.(it3.inf_pr, 1e-12); color = :forestgreen, linewidth = 2)
# Mark the constr_viol_tol target (1e-8 for these runs)
hlines!(ax_pr, [1e-8]; color = :black, linestyle = :dash, linewidth = 1.5)
text!(ax_pr, 0.02, 1e-8; text = "constr_viol_tol = 1e-8", space = :relative,
    align = (:left, :bottom), offset = (5, 2), fontsize = 13)
hidexdecorations!(ax_pr, grid = false)

ax_du = Axis(fig3[3, 1], xlabel = "Ipopt iteration", ylabel = "inf_du  (dual infeasibility)", yscale = log10)
lines!(ax_du, it2.iter, max.(it2.inf_du, 1e-12); color = :crimson,    linewidth = 2)
lines!(ax_du, it3.iter, max.(it3.inf_du, 1e-12); color = :forestgreen, linewidth = 2)

linkxaxes!(ax_obj, ax_pr, ax_du)

save(joinpath(@__DIR__, "ipopt_iter_history.png"), fig3)
println("Saved: ", joinpath(@__DIR__, "ipopt_iter_history.png"))
display(fig3)